# Lectura de un XML exportado desde una base de datos

Este ejemplo usa datos dummy, pero reproduce otro problema frecuente de ETL: cuando el CSV no se puede procesar, se pide XML como alternativa, pero el exportador corta las lineas cada 80 columnas y puede partir tags en el medio.

El archivo tiene estos problemas:

- no esta en UTF-8, sino en Windows-1252;
- el XML esta fisicamente cortado cada 80 caracteres;
- esos cortes pueden aparecer en medio de tags, por lo que `read_xml` no puede parsearlo;
- el codigo de producto sigue siendo un `varchar(12)` opaco que conviene convertir a hexadecimal.

In [1]:
from io import StringIO
from pathlib import Path
import pandas as pd

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")

if not raw_dir.exists():
    raw_dir = Path("etl-03/data/raw")
    processed_dir = Path("etl-03/data/processed")

archivo = raw_dir / "producto_dummy_problemas.xml"
archivo

PosixPath('data/raw/producto_dummy_problemas.xml')

## Qué es un `xml`

XML significa *eXtensible Markup Language*. Es un formato de texto usado para guardar o intercambiar datos de forma estructurada. A diferencia de un CSV, que organiza la información en filas y columnas separadas por comas, un XML usa **tags** o etiquetas para marcar qué significa cada dato.

Un XML se parece a HTML porque ambos usan etiquetas con apertura y cierre:

```xml
<Producto>
  <CodigoProducto>ABC123</CodigoProducto>
  <Nombre>Café molido</Nombre>
  <Precio>2450.50</Precio>
</Producto>
```

En ese ejemplo, `Producto` es el nodo principal y adentro tiene otros nodos: `CodigoProducto`, `Nombre` y `Precio`. Cada etiqueta encierra un valor.

También puede haber muchos productos dentro de un nodo raíz:

```xml
<Root>
  <Row>
    <CodigoProducto>ABC123</CodigoProducto>
    <Nombre>Café molido</Nombre>
  </Row>
  <Row>
    <CodigoProducto>DEF456</CodigoProducto>
    <Nombre>Té negro</Nombre>
  </Row>
</Root>
```

Para leerlo como tabla, pandas necesita encontrar un patrón repetido. En este notebook ese patrón es cada nodo `Row`: cada `Row` representa una fila y cada etiqueta interna se transforma en una columna.

La ventaja de XML es que conserva estructura y nombres de campos. La desventaja es que es más sensible a errores de formato: si falta una etiqueta de cierre, si el archivo está en otro encoding o si una línea corta un tag por la mitad, el parser puede fallar antes de llegar a los datos.


## 1. Primer intento con `pd.read_xml`

Si el archivo fuera XML valido, pandas podria leer cada nodo `Row` directamente. En este caso falla antes de llegar a los datos.

In [2]:
try:
    df_xml = pd.read_xml(archivo, xpath=".//Row", parser="etree")
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

UnicodeDecodeError
'utf-8' codec can't decode byte 0xe9 in position 65: invalid continuation byte


## 2. Probar encodings candidatos

Igual que con CSV, conviene separar dos problemas: primero decodificar bytes a texto; despues parsear la estructura.

In [3]:
encodings_candidatos = ["utf-8", "utf-8-sig", "cp1252", "latin1", "iso-8859-1"]

for encoding in encodings_candidatos:
    try:
        muestra = archivo.read_text(encoding=encoding)[:160]
    except UnicodeDecodeError as exc:
        print(f"{encoding:10} -> ERROR: {exc.reason}")
    else:
        print(f"{encoding:10} -> OK    : {muestra!r}")

utf-8      -> ERROR: invalid continuation byte
utf-8-sig  -> ERROR: invalid continuation byte
cp1252     -> OK    : '<Root><Row><CodigoProducto>8:#(,yo!c0</CodigoProducto><Nombre>Café molido clásic\no</Nombre><Categoria>Almacén</Categoria><Precio>2450.50</Precio><Notas>Paquete '
latin1     -> OK    : '<Root><Row><CodigoProducto>8:#(,yo!c0</CodigoProducto><Nombre>Café molido clásic\no</Nombre><Categoria>Almacén</Categoria><Precio>2450.50</Precio><Notas>Paquete '
iso-8859-1 -> OK    : '<Root><Row><CodigoProducto>8:#(,yo!c0</CodigoProducto><Nombre>Café molido clásic\no</Nombre><Categoria>Almacén</Categoria><Precio>2450.50</Precio><Notas>Paquete '


## 3. Diagnosticar el corte cada 80 columnas

El exportador agrego saltos fisicos de linea. Como son cortes mecanicos, algunas lineas terminan con un tag incompleto y la siguiente continua el mismo token XML.

In [5]:
texto = archivo.read_text(encoding="cp1252")
lineas = texto.splitlines()

print("Cantidad de lineas:", len(lineas))
print("Largos unicos de lineas:", sorted(set(map(len, lineas)))[:10], "...")

for numero, linea in enumerate(lineas, start=1):
    print(f"{numero:02d} ({len(linea):02d}): {linea!r}")

Cantidad de lineas: 18
Largos unicos de lineas: [8, 80] ...
01 (80): '<Root><Row><CodigoProducto>8:#(,yo!c0</CodigoProducto><Nombre>Café molido clásic'
02 (80): 'o</Nombre><Categoria>Almacén</Categoria><Precio>2450.50</Precio><Notas>Paquete d'
03 (80): 'e 500 g&#10;Origen: Brasil. Tostado medio.</Notas><Activo>1</Activo></Row><Row><'
04 (80): 'CodigoProducto>K7?,x9!!c0</CodigoProducto><Nombre>Té negro en hebras</Nombre><Ca'
05 (80): 'tegoria>Infusiones</Categoria><Precio>1320.00</Precio><Notas>Blend intenso con a'
06 (80): 'roma a bergamota.</Notas><Activo>1</Activo></Row><Row><CodigoProducto>M2$,ab@,90'
07 (80): '</CodigoProducto><Nombre>Jabón líquido para manos</Nombre><Categoria>Limpieza</C'
08 (80): 'ategoria><Precio>890.75</Precio><Notas>Fragancia limón.&#10;Apto dispenser.</Not'
09 (80): 'as><Activo>1</Activo></Row><Row><CodigoProducto>Z0%/,pq7!A</CodigoProducto><Nomb'
10 (80): 're>Aceite de oliva extra virgen</Nombre><Categoria>Almacén</Categoria><Precio>52'
11 (80): '00.00</Pre

Si intentamos parsear despues de corregir solo el encoding, el XML sigue roto porque los saltos quedaron dentro de tags o entidades.

In [6]:
try:
    pd.read_xml(StringIO(texto), xpath=".//Row", parser="etree")
except Exception as exc:
    print(type(exc).__name__)
    print(exc)

ParseError
not well-formed (invalid token): line 3, column 80


## 4. Reparar el wrapping fisico

En este archivo dummy, los saltos fisicos de linea son artefactos del exportador. Por eso podemos removerlos antes de parsear. En un caso real, esta decision requiere validar si tambien hay saltos de linea significativos dentro del texto.

In [8]:
xml_reparado = "".join(texto.splitlines())

print(xml_reparado[:300])

<Root><Row><CodigoProducto>8:#(,yo!c0</CodigoProducto><Nombre>Café molido clásico</Nombre><Categoria>Almacén</Categoria><Precio>2450.50</Precio><Notas>Paquete de 500 g&#10;Origen: Brasil. Tostado medio.</Notas><Activo>1</Activo></Row><Row><CodigoProducto>K7?,x9!!c0</CodigoProducto><Nombre>Té negro e


## 5. Leer el XML reparado

Ahora si pandas puede encontrar los nodos `Row`.

In [9]:
df = pd.read_xml(StringIO(xml_reparado), xpath=".//Row", parser="etree")
df

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo
0,"8:#(,yo!c0",Café molido clásico,Almacén,2450.50,Paquete de 500 g\nOrigen: Brasil. Tostado medio.,1
1,"K7?,x9!!c0",Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,1
2,"M2$,ab@,90",Jabón líquido para manos,Limpieza,890.75,Fragancia limón.\nApto dispenser.,1
3,"Z0%/,pq7!A",Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",1
4,"R8&,55#zP1",Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades.\nPuede contener...,0
5,"9V@,3D?,02",Yerba mate suave,Infusiones,3100.00,Producto estacionado. Elaboración argentina.,1


## 6. Tipos, notas y codigo hexadecimal

Ya con la estructura parseada, convertimos tipos y agregamos una version hexadecimal del codigo opaco.

In [10]:
df_limpio = df.copy()

df_limpio["Precio"] = pd.to_numeric(df_limpio["Precio"])
df_limpio["Activo"] = df_limpio["Activo"].map({1: True, 0: False, "1": True, "0": False})
df_limpio["Notas"] = df_limpio["Notas"].str.replace(r"\s*\r?\n\s*", " | ", regex=True)
df_limpio["CodigoProductoHex"] = df_limpio["CodigoProducto"].map(
    lambda valor: valor.encode("cp1252").hex(" ")
)

df_limpio

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo,CodigoProductoHex
0,"8:#(,yo!c0",Café molido clásico,Almacén,2450.50,Paquete de 500 g | Origen: Brasil. Tostado medio.,True,38 3a 23 28 2c 79 6f 21 63 30
1,"K7?,x9!!c0",Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,True,4b 37 3f 2c 78 39 21 21 63 30
2,"M2$,ab@,90",Jabón líquido para manos,Limpieza,890.75,Fragancia limón. | Apto dispenser.,True,4d 32 24 2c 61 62 40 2c 39 30
3,"Z0%/,pq7!A",Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",True,5a 30 25 2f 2c 70 71 37 21 41
4,"R8&,55#zP1",Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades. | Puede contene...,False,52 38 26 2c 35 35 23 7a 50 31
5,"9V@,3D?,02",Yerba mate suave,Infusiones,3100.00,Producto estacionado. Elaboración argentina.,True,39 56 40 2c 33 44 3f 2c 30 32


## 7. Validaciones simples

Validamos que las columnas principales no tengan nulos y que la columna hexadecimal exista.

In [11]:
columnas_esperadas = ["CodigoProducto", "Nombre", "Categoria", "Precio", "Notas", "Activo"]

assert list(df_limpio.columns[:len(columnas_esperadas)]) == columnas_esperadas
assert "CodigoProductoHex" in df_limpio.columns
assert not df_limpio["CodigoProducto"].isna().any()
assert not df_limpio["CodigoProductoHex"].isna().any()
assert not df_limpio["Nombre"].isna().any()

df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CodigoProducto     6 non-null      str    
 1   Nombre             6 non-null      str    
 2   Categoria          6 non-null      str    
 3   Precio             6 non-null      float64
 4   Notas              6 non-null      str    
 5   Activo             6 non-null      bool   
 6   CodigoProductoHex  6 non-null      str    
dtypes: bool(1), float64(1), str(5)
memory usage: 1.1 KB


## 8. Export opcional ya saneado

Guardamos el resultado normalizado en `data/processed`.

In [12]:
processed_dir.mkdir(parents=True, exist_ok=True)
salida = processed_dir / "producto_dummy_xml_limpio.csv"
df_limpio.to_csv(salida, index=False, encoding="utf-8")

pd.read_csv(salida)

,CodigoProducto,Nombre,Categoria,Precio,Notas,Activo,CodigoProductoHex
0,"8:#(,yo!c0",Café molido clásico,Almacén,2450.50,Paquete de 500 g | Origen: Brasil. Tostado medio.,True,38 3a 23 28 2c 79 6f 21 63 30
1,"K7?,x9!!c0",Té negro en hebras,Infusiones,1320.00,Blend intenso con aroma a bergamota.,True,4b 37 3f 2c 78 39 21 21 63 30
2,"M2$,ab@,90",Jabón líquido para manos,Limpieza,890.75,Fragancia limón. | Apto dispenser.,True,4d 32 24 2c 61 62 40 2c 39 30
3,"Z0%/,pq7!A",Aceite de oliva extra virgen,Almacén,5200.00,"Botella de vidrio. Acidez menor a 0,5%.",True,5a 30 25 2f 2c 70 71 37 21 41
4,"R8&,55#zP1",Galletas con chips de chocolate,Almacén,1750.25,Caja exhibidora x 12 unidades. | Puede contene...,False,52 38 26 2c 35 35 23 7a 50 31
5,"9V@,3D?,02",Yerba mate suave,Infusiones,3100.00,Producto estacionado. Elaboración argentina.,True,39 56 40 2c 33 44 3f 2c 30 32
